# Efficient LLM Serving at Scale with Unified Caching (vLLM + LMCache)

**The idea:** vLLM's prefix cache lives in GPU memory. When many long requests overflow that GPU pool, their KV gets **evicted and recomputed** on every repeat. LMCache adds a CPU-DRAM tier: evicted KV is **reloaded from host memory** instead of recomputed — cutting time-to-first-token (TTFT).

**We run the same overflow workload twice (cold then warm) under two servers:**

| Server | Warm-pass TTFT |
|---|---|
| vLLM prefix cache only | stays high — working set exceeds GPU pool, recomputed every time |
| vLLM prefix cache **+ LMCache** | **drops** — evicted KV reloaded from CPU |

Workload: 12 long requests (ISL 32K) at concurrency 4 — large enough to overflow the (deliberately small) GPU KV pool. Runs **inside `vllm/vllm-openai-rocm:v0.23.0`** on one MI300X.

## 1. Install LMCache + the AMD fixes

Stock image has no LMCache. Install it, then **force-reinstall the ROCm CuPy build** (a plain install can leave CuPy half-removed → `No module named cupy_backends`; ROCm CuPy is also what stops the cache server hanging at startup). The verify line must print `is_hip = True`.

In [ ]:
%%bash
set -e
pip install -q lmcache 2>&1 | tail -1 || true
pip uninstall -y cupy cupy-cuda12x cupy-cuda13x nixl nixl-cu12 nixl-cu13 nixl_ep >/dev/null 2>&1 || true
pip install --force-reinstall --no-cache-dir cupy-rocm-7-0 2>&1 | tail -2
pip install -q "numpy==2.1.3" "grpcio==1.78.0" 2>&1 | tail -1 || true
echo "---- verify (must show is_hip = True) ----"
python3 -c "import lmcache, cupy; from cupy_backends.cuda.api import runtime as r; print('lmcache', lmcache.__version__, '| cupy', cupy.__version__, '| is_hip =', getattr(r,'is_hip',False))" 

## 2. Baseline server — vLLM prefix cache only

`--gpu-memory-utilization 0.5` keeps the GPU KV pool small (~109K tokens) so the workload overflows it. `PYTHONHASHSEED=0` is required for cache-key consistency. Launches in the background; the next cell waits for it.

In [ ]:
%%bash
pkill -f "vllm serve" 2>/dev/null && sleep 4 || true
pkill -f "lmcache server" 2>/dev/null || true

export PYTHONHASHSEED=0
nohup vllm serve /models/gemma-4-31B-it \
  --served-model-name gemma-4-31B-it \
  --tensor-parallel-size 1 \
  --max-model-len 32768 \
  --gpu-memory-utilization 0.5 \
  --enable-prefix-caching \
  > /tmp/vllm.log 2>&1 &
echo "launching prefix-only server (log: /tmp/vllm.log); first boot ~4-6 min" 

## 3. Wait for the server (live log)

Polls `/v1/models` and prints the tail of the server log each attempt, so you can watch weight-load → compile → ready.

In [ ]:
import time, requests, subprocess
for i in range(80):
    try:
        if requests.get("http://localhost:8000/v1/models", timeout=5).ok:
            print("\n==> server ready"); break
    except requests.RequestException:
        pass
    tail = subprocess.run(["tail","-n","3","/tmp/vllm.log"], capture_output=True, text=True).stdout
    print(f"--- waiting {i+1}/80 ---\n{tail}", flush=True)
    time.sleep(10)
else:
    raise RuntimeError("server did not start - see /tmp/vllm.log")

## 4. Benchmark the baseline (two passes)

`vllm bench serve` with a fixed `--seed`, run twice: **PASS 1 (cold)** populates, **PASS 2 (warm)** re-sends the identical prompts. With prefix-cache only and a working set that overflows the GPU pool, **PASS 2 ≈ PASS 1** — no benefit, because evicted KV is recomputed.

In [ ]:
import subprocess, re

def bench():
    cmd = ["vllm","bench","serve",
        "--model","gemma-4-31B-it","--tokenizer","/models/gemma-4-31B-it",
        "--base-url","http://localhost:8000","--endpoint","/v1/completions",
        "--dataset-name","random","--random-input-len","32000","--random-output-len","64",
        "--num-prompts","12","--max-concurrency","4","--seed","555","--ignore-eos",
        "--percentile-metrics","ttft"]
    out = subprocess.run(cmd, capture_output=True, text=True).stdout
    m = re.search(r"Mean TTFT \(ms\):\s*([0-9.]+)", out)
    return float(m.group(1))/1000 if m else None

prefix_cold = bench(); print(f"prefix-only  PASS1 cold: {prefix_cold:6.2f} s")
prefix_warm = bench(); print(f"prefix-only  PASS2 warm: {prefix_warm:6.2f} s   ({prefix_cold/prefix_warm:.2f}x)")

## 5. Restart with LMCache — prefix cache + CPU KV offload

A separate `lmcache server` holds KV in CPU DRAM; vLLM talks to it via `LMCacheMPConnector`. **Both** processes get `PYTHONHASHSEED=0`. Same model, same GPU budget — only the CPU cache tier is added.

In [ ]:
%%bash
pkill -f "vllm serve" 2>/dev/null && sleep 4 || true
pkill -f "lmcache server" 2>/dev/null && sleep 2 || true

# CPU-DRAM KV cache server
export PYTHONHASHSEED=0
nohup lmcache server \
  --host 127.0.0.1 --port 5555 \
  --l1-size-gb 1000 --eviction-policy LRU --chunk-size 256 \
  > /tmp/lmcache_server.log 2>&1 &
sleep 6

# vLLM pointed at the LMCache server
export PYTHONHASHSEED=0
nohup vllm serve /models/gemma-4-31B-it \
  --served-model-name gemma-4-31B-it \
  --tensor-parallel-size 1 \
  --max-model-len 32768 \
  --gpu-memory-utilization 0.5 \
  --enable-prefix-caching \
  --kv-transfer-config '{"kv_connector":"LMCacheMPConnector","kv_role":"kv_both"}' \
  > /tmp/vllm.log 2>&1 &
echo "launching prefix+LMCache server; first boot ~4-6 min" 

## 6. Wait for the LMCache server

In [ ]:
for i in range(80):
    try:
        if requests.get("http://localhost:8000/v1/models", timeout=5).ok:
            print("\n==> server ready"); break
    except requests.RequestException:
        pass
    tail = subprocess.run(["tail","-n","3","/tmp/vllm.log"], capture_output=True, text=True).stdout
    print(f"--- waiting {i+1}/80 ---\n{tail}", flush=True)
    time.sleep(10)
else:
    raise RuntimeError("server did not start - see /tmp/vllm.log")

## 7. Benchmark with LMCache (same two passes)

Identical workload. Now **PASS 2 ≪ PASS 1** — evicted KV is reloaded from CPU instead of recomputed.

In [ ]:
lmcache_cold = bench(); print(f"LMCache  PASS1 cold: {lmcache_cold:6.2f} s")
lmcache_warm = bench(); print(f"LMCache  PASS2 warm: {lmcache_warm:6.2f} s   ({lmcache_cold/lmcache_warm:.2f}x)")

## 8. Proof it was the cache

The LMCache server logs `Stored` (cold) then `Retrieved` (warm). Only `Stored` and never `Retrieved` → cache keys mismatch (check `PYTHONHASHSEED=0` on both processes).

In [ ]:
%%bash
grep -iE "Stored [0-9]+ tokens|Retrieved [0-9]+ tokens" /tmp/lmcache_server.log | tail -6

## 9. Side-by-side result

Cold bars are similar (both recompute on first pass). The **warm bars are the story**: prefix-only stays high (recomputes evicted KV), prefix+LMCache drops (reloads from CPU).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

groups = ["Cold (PASS 1)", "Warm (PASS 2)"]
x = np.arange(2); w = 0.35
fig, ax = plt.subplots(figsize=(6.5, 4))
b1 = ax.bar(x - w/2, [prefix_cold, prefix_warm], w, label="prefix cache only", color="#c0504d")
b2 = ax.bar(x + w/2, [lmcache_cold, lmcache_warm], w, label="prefix + LMCache", color="#4f81bd")
ax.set_xticks(x); ax.set_xticklabels(groups)
ax.set_ylabel("Mean TTFT (s)")
ax.set_title("Warm pass: LMCache reloads evicted KV instead of recomputing")
ax.legend(); ax.grid(axis="y", alpha=0.3)
for bars in (b1, b2):
    for b in bars:
        ax.text(b.get_x()+b.get_width()/2, b.get_height(), f"{b.get_height():.1f}s",
                ha="center", va="bottom", fontsize=9)
plt.tight_layout(); plt.show()
print(f"Warm TTFT: prefix-only {prefix_warm:.1f}s vs LMCache {lmcache_warm:.1f}s "
      f"-> {prefix_warm/lmcache_warm:.1f}x faster")

## 10. When does LMCache help?

Only when **both** hold:

| Condition | Why |
|---|---|
| **Long context reused** | Reload only pays off if recompute (long prefill) is expensive. |
| **GPU HBM under pressure** | If the working set fits in GPU, vLLM's prefix cache already serves reuse and LMCache just adds transfer cost. |

That's why this demo overflows a small GPU pool with long requests. With a single short request, or a large GPU pool, prefix-cache alone is enough and LMCache shows no benefit — that's expected.

### AMD gotchas
- **`cupy-rocm-7-0` (force-reinstall)** — required, or the cache server hangs at startup.
- **`PYTHONHASHSEED=0` on every process** — required, or 0 % cache hits (silent).
- **`LMCacheMPConnector`** (separate server) — use on ROCm; the in-engine connector faults under concurrency.